## Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score, f1_score

## Load model results

In [ ]:
np.random.seed(42)

In [ ]:
df_mb = pd.read_csv("/content/gdrive/MyDrive/PhD_Models/saved_model/phd_model_iqa/mobilenetv2_validation.csv", index_col=False)
df_mb.sample(1)

,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa,label_pred,label_pred_proba_max,label_pred_proba
11499,GaussianBlur_7_f_frame_55.jpg,attack_client024_android_SD_iphone_video_scene01,attack,test,GaussianBlur,GaussianBlur_7,fas_iqa_dataset/GaussianBlur/test/attack/attac...,GaussianBlur_7,0.992465,"[1.3122310349444888e-08, 1.5936951314188263e-0..."


In [ ]:
df_ic = pd.read_csv("/content/gdrive/MyDrive/PhD_Models/saved_model/phd_model_iqa/InceptionResNetV2_validation.csv", index_col=False)
df_ic.sample(1)

,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa,label_pred,label_pred_proba_max,label_pred_proba
5603,547549.jpg,client007942_Env2_Ilum1_Spt6,attack,test,original,original,fas_iqa_dataset/original/test/attack/client007...,original,0.999885,"[5.997620733388942e-10, 2.45361425621482e-10, ..."


In [ ]:
df_rn = pd.read_csv("/content/gdrive/MyDrive/PhD_Models/saved_model/phd_model_iqa/resnet50_validation.csv", index_col=False)
df_rn.sample(1)

,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa,label_pred,label_pred_proba_max,label_pred_proba
14990,hbright_5_f_509138.jpg,client005567_Env0_Ilum0_Spt0,real,test,hbright,hbright_5,fas_iqa_dataset/hbright/test/real/client005567...,hbright_5,0.999949,"[5.544741110696805e-09, 1.7587094003346238e-09..."


## Load Class Dictionary

In [ ]:
class_dict = {0: 'BlurXY_11',
 1: 'BlurXY_15',
 2: 'BlurXY_7',
 3: 'BlurX_11',
 4: 'BlurX_15',
 5: 'BlurX_7',
 6: 'BlurY_11',
 7: 'BlurY_15',
 8: 'BlurY_7',
 9: 'GaussianBlur_11',
 10: 'GaussianBlur_15',
 11: 'GaussianBlur_7',
 12: 'hbright_10',
 13: 'hbright_5',
 14: 'hbright_8',
 15: 'jpgcompression_10',
 16: 'jpgcompression_30',
 17: 'jpgcompression_50',
 18: 'lbright_0.1',
 19: 'lbright_0.2',
 20: 'lbright_0.3',
 21: 'noise_15',
 22: 'noise_25',
 23: 'noise_45',
 24: 'original'}
class_dict_inv = {v:k for k,v in class_dict.items()}

## Evaluate Macro ACC and AUC

In [ ]:
results_dict = {}
dataset_name = ['MobileNetV2', 'Resnet50', 'InceptionResNetV2']
for idx, i_df in enumerate([df_mb, df_rn, df_ic]):
    label_iqa = i_df.label_iqa
    label_pred = i_df.label_pred
    label_pred_proba = i_df.label_pred_proba.apply(lambda x: eval(x))

    y_label = i_df.label_iqa.map(class_dict_inv).to_numpy()
    y_predict = i_df.label_pred.map(class_dict_inv).to_numpy()
    y_score = np.array(label_pred_proba.tolist())


    print(dataset_name[idx].upper())
    # print(classification_report(label_iqa, label_pred))
    # print(confusion_matrix(label_iqa, label_pred))
    print('ACC:', accuracy_score(y_label, y_predict))
    print('F1:', f1_score(y_label, y_predict, average='weighted'))
    print("Value ROC AUC:", roc_auc_score(y_label, y_score, multi_class='ovr'), '\n')

MOBILENETV2
ACC: 0.9592
F1: 0.9590397660093569
Value ROC AUC: 0.9990035879629628 

RESNET50
ACC: 0.9466666666666667
F1: 0.9464395758027543
Value ROC AUC: 0.9986699212962962 

INCEPTIONRESNETV2
ACC: 0.9565333333333333
F1: 0.9563553569806625
Value ROC AUC: 0.9991704189814814 



## Evaluate Class [Original] ACC and F1

In [ ]:
results_dict = {}
dataset_name = ['MobileNetV2', 'Resnet50', 'InceptionResNetV2']
for idx, i_df in enumerate([df_mb, df_rn, df_ic]):
    label_iqa = i_df.label_iqa
    label_pred = i_df.label_pred
    label_pred_proba = i_df.label_pred_proba.apply(lambda x: eval(x))

    y_label = i_df.label_iqa.map(class_dict_inv).to_numpy() == 24
    y_predict = i_df.label_pred.map(class_dict_inv).to_numpy() == 24
    y_score = np.array(label_pred_proba.tolist())


    print(dataset_name[idx].upper())
    print('ACC:', accuracy_score(y_label, y_predict))
    print('F1:', f1_score(y_label, y_predict, average='weighted'))
    # print("Value ROC AUC:", roc_auc_score(y_label.astype('uint8'), y_score, multi_class='ovr'), '\n')

MOBILENETV2
ACC: 0.9994
F1: 0.9993997602077979
RESNET50
ACC: 0.9982666666666666
F1: 0.9982666666666666
INCEPTIONRESNETV2
ACC: 0.999
F1: 0.9989963778636557


END